# ACCESS Metrics report for Purdue

In [ ]:
PROVIDER = 'Purdue'
RESOURCE_RENAMES = {
    'Purdue Anvil CPU': 'Anvil CPU',
    'Purdue Anvil GPU': 'Anvil GPU',
}

In [ ]:
YEAR = 2025
QUARTER_START_DATE_SUFFIX = '-01-01'
QUARTER_START_DATE = str(YEAR) + QUARTER_START_DATE_SUFFIX
QUARTER_END_DATE_SUFFIX = '-03-31'
QUARTER_END_DATE = str(YEAR) + QUARTER_END_DATE_SUFFIX
TWO_YEARS_AGO_QUARTER_START_DATE = str(YEAR - 2) + QUARTER_START_DATE_SUFFIX

In [ ]:
# This cell will be removed once JWT implementation is complete.
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path(os.path.expanduser('~/xdmod-data.env')), override=True)
os.environ['XDMOD_API_TOKEN'] = os.environ['PROD_API_TOKEN']

In [ ]:
#import sys
#! {sys.executable} -m pip install --upgrade 'xdmod-data>=1.0.0,<2.0.0' python-dotenv tabulate
#! {sys.executable} -m pip install  --upgrade xdmod-data
#! {sys.executable} -m pip install pandas
#! {sys.executable} -m pip show pandas

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import xdmod_data.themes
pio.templates.default = 'timeseries'
from xdmod_data.warehouse import DataWarehouse
from IPython.display import display, Markdown
try:
    pd.set_option('future.no_silent_downcasting',True)
except pd.errors.OptionError:
    pass
def display_df_md_table(df):
    return display(Markdown(df.replace('\n', '<br/>', regex=True).to_markdown(floatfmt=',.0f')))
dw = DataWarehouse('https://xdmod.access-ci.org')

In [ ]:
def create_plot(metric_label, resource, dimension_label, df,vertical_legend,nlargest):
        
        top_dimension_labels = None
        category_orders = None
        current_quarter_top_projects = []
        title = metric_label + (
            (f' on {resource}')
            if resource != 'all'
            else ''
        ) + f' by {dimension_label} by Quarter, Last Two Years'

        if nlargest > 0:
            
            list1 = df['Date'].unique().tolist()
            top_dimension_labels = []
            CURRENT_QUARTER = QUARTER_START_DATE + " 00:00:00"
                
            for date in list1:
                
                list2 = df[
                    df['Date'] == date
                ].nlargest(
                    nlargest,
                    metric_label,
                )[dimension_label].tolist()
                str_date = date.strftime('%Y-%m-%d %H:%M:%S')
    
                if str_date == CURRENT_QUARTER:
                    current_quarter_top_projects += list2
                    
                top_dimension_labels += list2
            top_dimension_labels = list(set(top_dimension_labels))
            df = df[df[dimension_label].isin(top_dimension_labels)]
            category_orders = {
                    dimension_label: top_dimension_labels,
                }
            title += f', Top {nlargest}'
            
            
            
        plot = px.line(
            df,
            x='Date',
            y=metric_label,
            title=title,
            color=dimension_label,
            markers=True,
            category_orders=category_orders,
        )
        plot.update_traces(
            hovertemplate='%{y:,.0f}',
        )
        plot.update_layout(
            xaxis_tickformat='Q%q %Y',
            hovermode='x unified',
            hoverlabel_namelength=-1,
        )
        if vertical_legend:
            plot.update_layout(
                legend_orientation='v',
                legend_xanchor='left',
                legend_x=0,
                legend_yanchor='bottom',
                legend_y=-1.3
            )
        plot.show()
        
        return (top_dimension_labels, current_quarter_top_projects)


def two_year_line_plot_by_quarter(
    y=None,
    resource=None,
    dimension=None,
    nlargest=0,
    vertical_legend=False
):
    if y == 'projects':
        metric = 'Number of Allocations: Active'
        metric_label = 'Number of Active Projects'
    elif y == 'users':
        metric = 'Number of Users: Active'
        metric_label = 'Number of Active Users'
    elif y == 'ace':
        metric = 'ACCESS Credit Equivalents Charged: Total (SU)'
        metric_label = 'ACCESS Credit Equivalents Charged'
    elif y == 'institutions':
        metric = 'Number of Institutions: Active'
        metric_label = 'Number of Institutions: Active'

    if resource == 'all':
        dimension = dimension_label = 'Resource'
        filters = {
            'Service Provider': PROVIDER,
        }
    else:
        filters = {
            'Resource': resource,
        }
    if dimension == 'pfos':
        dimension = 'Parent Science'
        dimension_label = 'Parent Field of Science'
    elif dimension == 'academic status':
        dimension = 'User NSF Status'
        dimension_label = 'User Academic Status'
    elif dimension == 'project':
        dimension = 'Allocation'
        dimension_label = 'Project'
    elif dimension == 'project type':
        realm = 'Allocations'
        dimension = 'Board Type'
        dimension_label = 'Project Type'
    with dw:
        df = dw.get_data(
            duration=(TWO_YEARS_AGO_QUARTER_START_DATE, QUARTER_END_DATE),
            realm=realm,
            metric=metric,
            dimension=dimension,
            dataset_type='timeseries',
            aggregation_unit='Quarter',
            filters=filters,
        )
    df = df.rename(
        columns={
            **RESOURCE_RENAMES,
            **{dimension: dimension_label}
        }
    )
    df = df.reset_index(names='Date')
    df = pd.melt(
        df,
        id_vars=['Date'],
        var_name=dimension_label,
        value_name=metric_label,
    
    )
    
    return create_plot(metric_label, resource, dimension_label, df,vertical_legend,nlargest)

## Active projects

### Total

In [ ]:
two_year_line_plot_by_quarter(
    y='projects',
    resource='all',
)

### By Project Type

In [ ]:
def plot_num_projects_by_type(resource):
    with dw:
        df=dw.get_data(
            realm='Jobs',
            dimension = 'Allocation',
            filters = {
                'Resource': resource,
                },
            duration = ('2023-01-01', '2025-06-30'),
            aggregation_unit='Quarter',
            metric = 'ACCESS Credit Equivalents Charged: Total (SU)'
        )

    list1 =df.columns.tolist()
    date_string_index = df.index.strftime('%Y-%m-%d')

    i=0
    while(i < 9):
        with dw:
           df2 =dw.get_raw_data(
               duration=(date_string_index[i],date_string_index[i+1]),
               realm='Allocations',
               fields = {'Name','Board Type'},
               filters = {'Resource': resource,}
            )
        filtered_df = df2[df2['Name'].isin(list1)].reset_index()
        filtered2_df = filtered_df.groupby(['Board Type']).count()
        filtered2_df = filtered2_df.reset_index()
        filtered2_df = filtered2_df.rename(columns={"Board Type": "Project Type", "Name": "Number of Active Projects"})
        filtered2_df = filtered2_df.drop(columns=['index'])
        filtered2_df.insert(0,"Date",date_string_index[i])
        if(i >= 1):
            filtered2_df = pd.concat([previous_df,filtered2_df])
        
        previous_df = filtered2_df
        i+=1
    create_plot(metric_label = 'Number of Active Projects', resource=resource, 
        dimension_label='Project Type', df=filtered2_df,vertical_legend = False,nlargest=0)

In [ ]:
plot_num_projects_by_type('Purdue Anvil GPU')
plot_num_projects_by_type('Purdue Anvil CPU')

### Some Projects started as one type but later changed to a different type

### By Parent Field of Science

In [ ]:
two_year_line_plot_by_quarter(
    y='projects',
    resource='Purdue Anvil CPU',
    dimension='pfos',
)

In [ ]:
two_year_line_plot_by_quarter(
    y='projects',
    resource='Purdue Anvil GPU',
    dimension='pfos',
)

In [ ]:
top_projects , current_quarter_top_projects = two_year_line_plot_by_quarter(
    y='ace',
    resource='Purdue Anvil CPU',
    dimension='project',
    nlargest=5,
    vertical_legend=True,
)

In [ ]:
dimensions = ['PI', 'Parent Science']
aces = []
dimension_counts = {}

with dw:
    for dimension in dimensions:
        dimension_counts[dimension] = []
        for project in current_quarter_top_projects:
            
            
            df = dw.get_data(
                duration=(QUARTER_START_DATE, QUARTER_END_DATE),
                realm='Jobs',
                metric='ACCESS Credit Equivalents Charged: Total (SU)',
                dimension=dimension,
                dataset_type='aggregate',
                aggregation_unit='quarter',
                filters={
                    'Allocation': project,
                    'Resource': 'Purdue Anvil CPU',
                },
            )
            
            if not df.empty: 
                dimension_counts[dimension].append(df.index[0])
                if dimension == dimensions[0]:
                    aces.append(df.iloc[0])

In [ ]:
data = [
    current_quarter_top_projects,
    aces,
]

for dimension_values in dimension_counts.values():
    data.append(dimension_values)
df = pd.DataFrame(data).transpose()
df.columns = ['Projects of ' + QUARTER_START_DATE, 'ACEs Charged'] + ['PI', 'Parent Field of Science']

display_df_md_table(df)


## Active users

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='all',
    dimension=None,
)

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil CPU',
    dimension='pfos',
)

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil GPU',
    dimension='pfos',
)

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil CPU',
    dimension='academic status',
)

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil GPU',
    dimension='academic status',
)

In [ ]:
two_year_line_plot_by_quarter(
    y='users',
    resource='Purdue Anvil GPU',
    dimension='academic status',
)

In [ ]:
with dw:
    metrics = dw.describe_metrics('Jobs')
display_df_md_table(metrics)

## Active Users Institutions

In [ ]:
two_year_line_plot_by_quarter(
    y='institutions',
    resource='all',
    dimension=None,
)

In [ ]:
two_year_line_plot_by_quarter(
    y='ace',
    resource='all',
    dimension='pfos',
)